# Notebook 3 – Segmentation Classique

**PFE 2025/2026** – Segmentation d'images en présence de bruit  

---

Méthodes étudiées :
1. **Filtre de Canny** – détection de contours
2. **Seuillage d'Otsu** – segmentation par seuil global optimal
3. **Seuillage adaptatif** – seuillage local
4. **Watershed** – segmentation par lignes de partage des eaux
5. **Robustesse** – impact du bruit sur chaque méthode

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from pathlib import Path
from noise import add_gaussian_noise, add_salt_and_pepper_noise
from denoising import denoise
from segmentation.classical import canny_segmentation, otsu_segmentation, watershed_segmentation
from segmentation.classical import adaptive_threshold_segmentation

print('Modules chargés ✓')

In [ ]:
# Charger image
data_dir = Path('../data/images')
imgs = list(data_dir.glob('*.png')) + list(data_dir.glob('*.jpg'))

if imgs:
    image = cv2.imread(str(imgs[0]))
else:
    image = np.zeros((256, 256, 3), dtype=np.uint8)
    cv2.circle(image, (128, 128), 80, (200, 200, 200), -1)
    cv2.rectangle(image, (40, 40), (100, 100), (80, 80, 80), -1)
    cv2.putText(image, 'TEST', (90, 230), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (150, 150, 150), 2)

print(f'Shape: {image.shape}')

## 3.1 Comparaison des méthodes sur image propre

In [ ]:
_, watershed_overlay = watershed_segmentation(image)

results_clean = {
    'Canny':       canny_segmentation(image),
    'Otsu':        otsu_segmentation(image),
    'Adaptatif':   adaptive_threshold_segmentation(image),
    'Watershed':   watershed_overlay,
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')

for ax, (name, mask) in zip(axes[1:], results_clean.items()):
    if mask.ndim == 3:
        ax.imshow(cv2.cvtColor(mask, cv2.COLOR_BGR2RGB))
    else:
        ax.imshow(mask, cmap='gray')
    ax.set_title(name)
    ax.axis('off')

plt.suptitle('Segmentation sur image propre', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/segmentation_propre.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.2 Robustesse au bruit Gaussien – Canny

In [ ]:
sigmas = [0, 10, 25, 50, 75]

fig, axes = plt.subplots(3, len(sigmas), figsize=(16, 9))

for col, sigma in enumerate(sigmas):
    noisy = add_gaussian_noise(image, sigma=sigma) if sigma > 0 else image.copy()
    filtered = denoise(noisy, 'median')
    
    label = f'σ={sigma}' if sigma > 0 else 'Original'
    
    axes[0, col].imshow(cv2.cvtColor(noisy, cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(label, fontsize=9)
    axes[0, col].axis('off')
    
    axes[1, col].imshow(canny_segmentation(noisy), cmap='gray')
    axes[1, col].set_title('Canny brut', fontsize=9) if col == 0 else axes[1, col].set_title('', fontsize=9)
    axes[1, col].axis('off')
    
    axes[2, col].imshow(canny_segmentation(filtered), cmap='gray')
    axes[2, col].set_title('Médian+Canny', fontsize=9) if col == 0 else axes[2, col].set_title('', fontsize=9)
    axes[2, col].axis('off')

plt.suptitle('Robustesse de Canny au bruit Gaussien (avec/sans pré-filtrage)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/canny_robustesse.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 Robustesse au bruit Gaussien – Otsu

In [ ]:
fig, axes = plt.subplots(3, len(sigmas), figsize=(16, 9))

for col, sigma in enumerate(sigmas):
    noisy = add_gaussian_noise(image, sigma=sigma) if sigma > 0 else image.copy()
    filtered = denoise(noisy, 'median')
    
    label = f'σ={sigma}' if sigma > 0 else 'Original'
    axes[0, col].imshow(cv2.cvtColor(noisy, cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(label, fontsize=9)
    axes[0, col].axis('off')
    
    axes[1, col].imshow(otsu_segmentation(noisy), cmap='gray')
    axes[1, col].set_title('Otsu brut', fontsize=9) if col == 0 else None
    axes[1, col].axis('off')
    
    axes[2, col].imshow(otsu_segmentation(filtered), cmap='gray')
    axes[2, col].set_title('Médian+Otsu', fontsize=9) if col == 0 else None
    axes[2, col].axis('off')

plt.suptitle('Robustesse de Otsu au bruit Gaussien', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/otsu_robustesse.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 Algorithme de Canny – Explication étape par étape

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Étape 1: Lissage Gaussien
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# Étape 2: Gradient (Sobel)
grad_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
grad_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
gradient = np.sqrt(grad_x**2 + grad_y**2)
gradient = (gradient / gradient.max() * 255).astype(np.uint8)

# Étape 3: Canny complet
edges_low  = cv2.Canny(blurred, 30,  90)
edges_mid  = cv2.Canny(blurred, 50, 150)
edges_high = cv2.Canny(blurred, 100, 200)

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, img, title in zip(axes,
    [gray, blurred, gradient, edges_mid, edges_high],
    ['Niveaux de gris', '1. Lissage', '2. Gradient', 'Canny (50,150)', 'Canny (100,200)']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.suptitle('Pipeline du filtre de Canny', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/canny_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()